<a href="https://colab.research.google.com/github/Hadishof/AnanLabs/blob/main/StudentsForm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
import os

FILE_PATH = '/content/drive/MyDrive/מחשוב ענן/תרגול 1/Students.txt' #@param {type:"string"}

if os.path.exists(FILE_PATH):
    print('Found:', FILE_PATH)
    with open(FILE_PATH, 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print(' File not found:', FILE_PATH)
    folder = os.path.dirname(FILE_PATH)
    if os.path.isdir(folder):
        print('Files in', folder, '->', os.listdir(folder))
    else:
        print('Folders in My Drive ->', os.listdir('/content/drive/MyDrive'))

Found: /content/drive/MyDrive/מחשוב ענן/תרגול 1/Students.txt
Student Information

First Name: Tali
Last Name: Hamoud
Braude Email: Talihamoud@e.braude.ac.il

Courses This Semester:
WEB

Interesting Link:

***************************
First Name: Hadi
Last Name: Shofania
Braude Email: HadiShofania@e.braude.ac.il

Courses This Semester:
WEB

Interesting Link:

***************************
First Name: Yaren
Last Name: Kablan
Braude Email: YarenKablan@e.braude.ac.il

Courses This Semester:
WEB

Interesting Link:

***************************
First Name: Lina
Last Name: Shannan
Braude Email: LinaShnnan@e.braude.ac.il

Courses This Semester:
WEB

Interesting Link:



In [49]:
#@title 3. Student form
import ipywidgets as widgets
from IPython.display import display, HTML
from datetime import datetime

# ---------- File handling ----------
SEPARATOR = '***************************'
KEYS = ('first name', 'last name', 'braude email',
        'courses this semester', 'interesting link', 'favorite program')

def _clean_email(value):
    """Handles both 'a@b.c' and markdown '[a@b.c](mailto:a@b.c)'."""
    value = value.strip()
    if value.startswith('[') and '](' in value:
        value = value[1:value.index('](')]
    return value.strip()

def _is_sep(line):
    return line.strip() != '' and set(line.strip()) == {'*'}

def _read_lines(path):
    with open(path, 'r', encoding='utf-8') as f:
        return f.read().split('\n')

def _write_lines(path, lines):
    with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))

def _block_ranges(lines):
    """Return [(start, end), ...] for every block between separator lines."""
    blocks, start = [], 0
    for i, l in enumerate(lines):
        if _is_sep(l):
            blocks.append((start, i))
            start = i + 1
    blocks.append((start, len(lines)))
    return blocks

def _find_block(lines, email):
    """Range of the (last) block that belongs to this email."""
    target = None
    for b_start, b_end in _block_ranges(lines):
        for l in lines[b_start:b_end]:
            k, sep, v = l.partition(':')
            if sep and k.strip().lower() == 'braude email' and \
               _clean_email(v).lower() == email.lower():
                target = (b_start, b_end)
    if target is None:
        raise ValueError(f'Student {email} not found in file')
    return target

def load_students(path):
    """Parse Students.txt into {email_lower: student_dict}."""
    lines = _read_lines(path)
    students = {}
    for b_start, b_end in _block_ranges(lines):
        s = {'first': '', 'last': '', 'email': '', 'courses': [],
             'link': '', 'favorites': []}
        in_courses = False
        for raw in lines[b_start:b_end]:
            line = raw.strip()
            if not line:
                continue
            key, sep, value = line.partition(':')
            k, value = key.strip().lower(), value.strip()
            if sep and k in KEYS:
                in_courses = (k == 'courses this semester')
                if k == 'first name':         s['first'] = value
                elif k == 'last name':        s['last'] = value
                elif k == 'braude email':     s['email'] = _clean_email(value)
                elif k == 'interesting link': s['link'] = value
                elif k == 'favorite program' and value:
                    s['favorites'].append(value)
                elif in_courses and value:    s['courses'].append(value)
            elif in_courses:
                s['courses'].append(line)
            elif line.lower().startswith('http') and not s['link']:
                s['link'] = line            # link written on the line after the label
        if s['email'] or s['first']:
            students[(s['email'] or f"{s['first']} {s['last']}").lower()] = s
    return students

def add_favorite(path, email, favorite):
    """Insert 'Favorite Program: X' inside the student's block,
    under the last Favorite Program line (or under Interesting Link)."""
    lines = _read_lines(path)
    b_start, b_end = _find_block(lines, email)
    insert_at = None
    for i in range(b_start, b_end):
        k = lines[i].partition(':')[0].strip().lower()
        if k == 'favorite program':
            insert_at = i + 1
        elif k == 'interesting link' and insert_at is None:
            insert_at = i + 1
            if i + 1 < b_end and lines[i + 1].strip().lower().startswith('http'):
                insert_at = i + 2
    if insert_at is None:
        insert_at = b_end
        while insert_at > b_start and lines[insert_at - 1].strip() == '':
            insert_at -= 1
    lines.insert(insert_at, f'Favorite Program: {favorite}')
    _write_lines(path, lines)

def delete_favorite(path, email, favorite):
    """Remove one 'Favorite Program: X' line from the student's block."""
    lines = _read_lines(path)
    b_start, b_end = _find_block(lines, email)
    for i in range(b_end - 1, b_start - 1, -1):
        k, sep, v = lines[i].partition(':')
        if sep and k.strip().lower() == 'favorite program' and v.strip() == favorite:
            del lines[i]
            _write_lines(path, lines)
            return True
    return False

# ---------- Styling ----------
display(HTML("""
<style>
.sf-card {
  background: linear-gradient(135deg, #1e3a5f 0%, #2d6a9f 100%);
  color: #fff; padding: 18px 22px; border-radius: 14px;
  font-family: 'Segoe UI', Arial, sans-serif; margin-bottom: 10px;
  box-shadow: 0 4px 14px rgba(0,0,0,.18);
}
.sf-card h2 { margin: 0 0 4px 0; font-size: 22px; }
.sf-card p  { margin: 0; opacity: .85; font-size: 13px; }
.sf-link a  { color: #2d6a9f; font-weight: 600; text-decoration: none; }
.sf-link a:hover { text-decoration: underline; }
</style>
<div class="sf-card">
  <h2>🎓 Braude Students – Cloud Computing</h2>
  <p>Select a student to view their details, then add or delete favorite programs.</p>
</div>
"""))

# ---------- Widgets ----------
FIELD_LAYOUT = widgets.Layout(width='460px')
STYLE = {'description_width': '130px'}
BTN_MARGIN = '8px 0 0 136px'

students = load_students(FILE_PATH)

student_dd = widgets.Dropdown(
    options=[(f"{s['first']} {s['last']}", key) for key, s in students.items()],
    description='Student:', style=STYLE, layout=FIELD_LAYOUT)

first_tb   = widgets.Text(description='First name:',   disabled=True, style=STYLE, layout=FIELD_LAYOUT)
last_tb    = widgets.Text(description='Last name:',    disabled=True, style=STYLE, layout=FIELD_LAYOUT)
email_tb   = widgets.Text(description='Braude email:', disabled=True, style=STYLE, layout=FIELD_LAYOUT)
courses_ta = widgets.Textarea(description='Courses:',  disabled=True, style=STYLE,
                              layout=widgets.Layout(width='460px', height='70px'))
link_tb    = widgets.Text(description='Interesting link:', disabled=True, style=STYLE, layout=FIELD_LAYOUT)
link_html  = widgets.HTML()
favs_ta    = widgets.Textarea(description='Favorite programs:', disabled=True, style=STYLE,
                              layout=widgets.Layout(width='460px', height='70px'))

# add
new_fav_tb = widgets.Text(description='New program:', placeholder='e.g. Breaking Bad',
                          style=STYLE, layout=FIELD_LAYOUT)
update_btn = widgets.Button(description='Update', button_style='success', icon='save',
                            layout=widgets.Layout(width='140px', margin=BTN_MARGIN))
# delete
del_dd     = widgets.Dropdown(description='Program:', style=STYLE, layout=FIELD_LAYOUT)
delete_btn = widgets.Button(description='Delete', button_style='danger', icon='trash',
                            layout=widgets.Layout(width='140px', margin=BTN_MARGIN))

status_out = widgets.Output()

# ---------- Behaviour ----------
def show_student(key):
    s = students.get(key)
    if not s:
        return
    first_tb.value   = s['first']
    last_tb.value    = s['last']
    email_tb.value   = s['email']
    courses_ta.value = '\n'.join(s['courses']) or '—'
    link_tb.value    = s['link'] or '—'
    link_html.value  = (f'<div class="sf-link" style="margin-left:136px">'
                        f'<a href="{s["link"]}" target="_blank">🔗 Open link</a></div>'
                        if s['link'] else '')
    favs_ta.value    = '\n'.join(s['favorites']) or '—'
    del_dd.options   = s['favorites']
    del_dd.value     = s['favorites'][-1] if s['favorites'] else None
    delete_btn.disabled = not s['favorites']
    new_fav_tb.value = ''

def reload_and_show(key):
    global students
    students = load_students(FILE_PATH)
    show_student(key)

def on_student_change(change):
    if change['new']:
        status_out.clear_output()
        show_student(change['new'])

def on_update_clicked(_):
    status_out.clear_output()
    update_btn.disabled = True                 # prevents double clicks
    try:
        with status_out:
            key = student_dd.value
            new_fav = new_fav_tb.value.strip()
            if not key:
                print('⚠️ No student selected.')
                return
            if not new_fav:
                print('⚠️ Please type a program first.')
                return
            s = students[key]
            if new_fav.lower() in (f.lower() for f in s['favorites']):
                print(f"ℹ️ '{new_fav}' is already in {s['first']}'s list.")
                return
            add_favorite(FILE_PATH, s['email'], new_fav)
            reload_and_show(key)
            print(f"✅ {datetime.now():%H:%M:%S} – added '{new_fav}' for {s['first']} {s['last']}")
    finally:
        update_btn.disabled = False

def on_delete_clicked(_):
    status_out.clear_output()
    delete_btn.disabled = True
    try:
        with status_out:
            key = student_dd.value
            fav = del_dd.value
            if not key or not fav:
                print('⚠️ Nothing to delete.')
                return
            s = students[key]
            if delete_favorite(FILE_PATH, s['email'], fav):
                reload_and_show(key)
                print(f"🗑️ {datetime.now():%H:%M:%S} – deleted '{fav}' for {s['first']} {s['last']}")
            else:
                print(f"⚠️ '{fav}' was not found in the file.")
    finally:
        s = students.get(student_dd.value)
        delete_btn.disabled = not (s and s['favorites'])

student_dd.observe(on_student_change, names='value')
update_btn.on_click(on_update_clicked)
delete_btn.on_click(on_delete_clicked)

# ---------- Layout ----------
details_box = widgets.VBox(
    [first_tb, last_tb, email_tb, courses_ta, link_tb, link_html, favs_ta],
    layout=widgets.Layout(border='1px solid #c7d6e6', border_radius='10px',
                          padding='12px', margin='8px 0'))
add_box = widgets.VBox(
    [widgets.HTML('<b>➕ Add favorite program</b>'), new_fav_tb, update_btn],
    layout=widgets.Layout(border='1px solid #b9e0c3', border_radius='10px',
                          padding='12px', margin='0 0 8px 0'))
del_box = widgets.VBox(
    [widgets.HTML('<b>🗑️ Delete favorite program</b>'), del_dd, delete_btn],
    layout=widgets.Layout(border='1px solid #f0c2c2', border_radius='10px', padding='12px'))

display(widgets.VBox([student_dd, details_box, add_box, del_box, status_out]))

if student_dd.value:
    show_student(student_dd.value)


In [50]:
#@title 4. Show file contents (check the appended block)
with open(FILE_PATH, 'r', encoding='utf-8') as f:
    print(f.read())


Student Information

First Name: Tali
Last Name: Hamoud
Braude Email: Talihamoud@e.braude.ac.il

Courses This Semester:
WEB

Interesting Link:

***************************
First Name: Hadi
Last Name: Shofania
Braude Email: HadiShofania@e.braude.ac.il

Courses This Semester:
WEB

Interesting Link:

***************************
First Name: Yaren
Last Name: Kablan
Braude Email: YarenKablan@e.braude.ac.il

Courses This Semester:
WEB

Interesting Link:

***************************
First Name: Lina
Last Name: Shannan
Braude Email: LinaShnnan@e.braude.ac.il

Courses This Semester:
WEB

Interesting Link:

